# SIM V1 3D — Phase C: train the volumetric path-loss surrogate

3-D analog of `SIM/phase_c_train_colab_v3.ipynb`. Learns a UNet that maps
**(fixed geometry + Tx + frequency) → PL(x,y,z)** so the browser can redraw the
volume in one forward pass instead of re-running the ray-march.

**Inputs (9 channels, same scheme as the 2-D pipeline):** 6 material one-hot
(fixed geometry) + Tx Gaussian blob + frequency feature + log10-distance.
**Target:** normalized PL volume from Phase B-3D.

Upload the `SIM V1 3D/` folder (with `dataset/` shards, `manifest_3d.json`,
`material_grid.npy`, `inside_mask.npy`) to Drive and point `ROOT` at it.


In [ ]:
import os, json, glob, numpy as np, torch, torch.nn as nn, torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

ROOT = "/content/drive/MyDrive/SIM V1 3D"   # <-- edit to your upload path
dev = "cuda" if torch.cuda.is_available() else "cpu"
print("device:", dev, "| torch", torch.__version__)

man = json.load(open(f"{ROOT}/manifest_3d.json"))
M = np.load(f"{ROOT}/material_grid.npy")          # (nx,ny,nz) int8
inside = np.load(f"{ROOT}/inside_mask.npy")       # (nx,ny,nz) bool
nx, ny, nz = M.shape
C = len(man["materials"])
norm = man["norm"]; cell = man["cell_size_m"]
SIGMA, DNORM = 2.0, 3.0        # Tx blob sigma (cells), log-dist normaliser (2-D parity)
print("grid", (nx, ny, nz), "| classes", C, "| interior voxels", int(inside.sum()))

## Input featurization

The geometry is fixed, so the material one-hot is a constant tensor; only the Tx
blob, the distance channel, and the (broadcast) frequency feature change per
sample. Built on the fly from the `(tx, freq_feat)` stored by Phase B-3D.

In [ ]:
# constant geometry channels (6, nx, ny, nz)
onehot = np.stack([(M == c) for c in range(C)], 0).astype(np.float32)
onehot_t = torch.from_numpy(onehot)
gx, gy, gz = np.meshgrid(np.arange(nx), np.arange(ny), np.arange(nz), indexing="ij")
grid_xyz = np.stack([gx, gy, gz], 0).astype(np.float32)   # (3,nx,ny,nz)

def make_input(tx, freq_feat):
    d2 = ((grid_xyz - np.asarray(tx, np.float32)[:, None, None, None]) ** 2).sum(0)
    blob = np.exp(-d2 / (2 * SIGMA ** 2)).astype(np.float32)
    d_m = np.maximum(np.sqrt(d2) * cell, 1.0)
    logd = (np.log10(d_m) / DNORM).astype(np.float32)
    ff = np.full((nx, ny, nz), float(freq_feat), np.float32)
    dyn = np.stack([blob, ff, logd], 0)                   # (3,nx,ny,nz)
    return torch.from_numpy(np.concatenate([onehot, dyn], 0))  # (9,...)

CIN = C + 3

In [ ]:
class ShardDS(Dataset):
    """Flattens the Phase B-3D shards into (tx, freq_feat, target) samples,
    restricted to a split (list of pos_ids)."""
    def __init__(self, root, keep_pos):
        self.items = []
        keep = set(keep_pos)
        for sp in sorted(glob.glob(f"{root}/dataset/shard_*.npz")):
            d = np.load(sp)
            for i in range(len(d["pos_id"])):
                if int(d["pos_id"][i]) in keep:
                    self.items.append((d["tx"][i].copy(), float(d["freq_feat"][i]),
                                       d["target"][i].astype(np.float32).copy()))
    def __len__(self): return len(self.items)
    def __getitem__(self, i):
        tx, ff, tgt = self.items[i]
        return make_input(tx, ff), torch.from_numpy(tgt)[None]   # x:(9,..) y:(1,..)

splits = json.load(open(f"{ROOT}/dataset/splits.json"))
tr = ShardDS(ROOT, splits["train"]); va = ShardDS(ROOT, splits["val"]); te = ShardDS(ROOT, splits["test"])
print("samples  train", len(tr), "val", len(va), "test", len(te))
BATCH = 2
tl = DataLoader(tr, BATCH, shuffle=True); vl = DataLoader(va, BATCH); tel = DataLoader(te, BATCH)

## 3-D UNet

Anisotropic pooling `(2,1,2)` — the vertical axis (`ny≈11`) is thin, so we pool
only X and Z. Skips are size-matched with `interpolate`, which tolerates the odd
dimensions that halving 262/118 produces.

In [ ]:
def dconv(ci, co):
    return nn.Sequential(nn.Conv3d(ci, co, 3, padding=1), nn.BatchNorm3d(co), nn.ReLU(inplace=True),
                         nn.Conv3d(co, co, 3, padding=1), nn.BatchNorm3d(co), nn.ReLU(inplace=True))

class UNet3D(nn.Module):
    def __init__(self, cin=CIN, base=16):
        super().__init__()
        self.pool = nn.MaxPool3d((2, 1, 2))
        self.e1 = dconv(cin, base); self.e2 = dconv(base, base * 2)
        self.b = dconv(base * 2, base * 4)
        self.d2 = dconv(base * 4 + base * 2, base * 2)
        self.d1 = dconv(base * 2 + base, base)
        self.out = nn.Conv3d(base, 1, 1)

    def up(self, x, skip):
        x = F.interpolate(x, size=skip.shape[2:], mode="trilinear", align_corners=False)
        return torch.cat([x, skip], 1)

    def forward(self, x):
        e1 = self.e1(x); e2 = self.e2(self.pool(e1)); b = self.b(self.pool(e2))
        d2 = self.d2(self.up(b, e2)); d1 = self.d1(self.up(d2, e1))
        return torch.sigmoid(self.out(d1))       # normalized PL in [0,1]

model = UNet3D().to(dev)
mask = torch.from_numpy(inside.astype(np.float32))[None, None].to(dev)   # (1,1,...)
print(sum(p.numel() for p in model.parameters()) / 1e6, "M params")

In [ ]:
def masked_mse(pred, tgt):
    m = mask.expand_as(pred)
    return ((pred - tgt) ** 2 * m).sum() / m.sum().clamp(min=1)

opt = torch.optim.Adam(model.parameters(), 1e-3)
EPOCHS = 40
for ep in range(EPOCHS):
    model.train(); tot = 0.0
    for x, y in tl:
        x, y = x.to(dev), y.to(dev)
        opt.zero_grad(); loss = masked_mse(model(x), y); loss.backward(); opt.step()
        tot += loss.item() * len(x)
    model.eval(); vtot = 0.0
    with torch.no_grad():
        for x, y in vl:
            x, y = x.to(dev), y.to(dev); vtot += masked_mse(model(x), y).item() * len(x)
    print(f"ep {ep+1:02d}  train {tot/len(tr):.4f}  val {vtot/max(len(va),1):.4f}")

## Test metrics in dB + physics baselines

Denormalize to dB and report RMSE/MAE over interior voxels, against FSPL and
log-distance baselines (as in the 2-D Phase C).

In [ ]:
PLR, PLO = norm["pl_range_db"], norm["pl_min_db"]
def to_db(v): return v * PLR + PLO

def rmse_mae(pred, tgt):
    m = inside
    e = (to_db(pred) - to_db(tgt))[m]
    return float(np.sqrt((e ** 2).mean())), float(np.abs(e).mean())

model.eval(); preds, tgts = [], []
with torch.no_grad():
    for x, y in tel:
        preds.append(model(x.to(dev)).cpu().numpy()[:, 0]); tgts.append(y.numpy()[:, 0])
preds = np.concatenate(preds); tgts = np.concatenate(tgts)
r, mae = rmse_mae(preds, tgts)
print(f"surrogate  RMSE {r:.2f} dB   MAE {mae:.2f} dB   (over interior, test split)")

# FSPL baseline: predict free space only (needs tx/freq per test sample) —
# reconstruct from stored items for a like-for-like comparison.
fspl_c = man["physics"]["fspl_const_db"]; n_exp = man["physics"]["n_exp"]
f_lo, f_hi = np.log10(norm["freq_log_lo_mhz"]), np.log10(norm["freq_log_hi_mhz"])
errs = []
for (tx, ff, tgt) in te.items:
    f_mhz = 10 ** (ff * (f_hi - f_lo) + f_lo)
    d2 = ((grid_xyz - np.asarray(tx, np.float32)[:, None, None, None]) ** 2).sum(0)
    d_m = np.maximum(np.sqrt(d2) * cell, 1.0)
    pl_fspl = 20 * np.log10(f_mhz) + fspl_c + 10 * n_exp * np.log10(d_m)
    errs.append((pl_fspl - to_db(tgt.astype(np.float32)))[inside])
errs = np.concatenate(errs)
print(f"FSPL only  RMSE {np.sqrt((errs**2).mean()):.2f} dB   MAE {np.abs(errs).mean():.2f} dB")

## Export to ONNX

Fixed input shape `(1, 9, nx, ny, nz)`. onnxruntime-web runs 3-D `Conv` on the
**wasm** backend (the WebGPU EP does not yet cover Conv3d). Then run
`export_web3.py` to bundle the grid/masks + `manifest_3d` as `sim_assets_3d.js`.

In [ ]:
model.eval()
dummy = torch.zeros(1, CIN, nx, ny, nz, device=dev)
torch.onnx.export(model, dummy, f"{ROOT}/web/pl_unet3d.onnx",
                  input_names=["x"], output_names=["pl"], opset_version=17)
sz = os.path.getsize(f"{ROOT}/web/pl_unet3d.onnx") / 1e6
print(f"wrote pl_unet3d.onnx ({sz:.1f} MB)")